<a href="https://colab.research.google.com/github/SamiSayem13/Zunayra-Ai/blob/main/Zunayra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -U transformers accelerate

In [ ]:
!pip install -q tavily-python

In [ ]:
from tavily import TavilyClient
from google.colab import userdata

def web_search(query):
    """
    Performs a web search using the Tavily API.
    """
    try:
        # Correctly accessing the secret named 'TAVILY_API_KEY'
        tavily_key = userdata.get('TAVILY_API_KEY')
        tavily = TavilyClient(api_key=tavily_key)
        # Using search_depth='advanced' for better cross-checking
        search_result = tavily.search(query=query, search_depth="advanced", max_results=5)

        context = ""
        for result in search_result['results']:
            context += f"Source: {result['url']}\nContent: {result['content']}\n\n"
        return context
    except Exception as e:
        return f"Error performing search: {str(e)}"

In [ ]:
import platform

print(platform.system())
print(platform.release())
print(platform.node())

In [ ]:
# Used to securely store your API key
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
"""
Need HF_Token from the huggingface.
"""
model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    token=HF_TOKEN
)

In [ ]:
def chat(user_message, system_content=None):
    current_memory_state = load_memory()
    ai_nickname = current_memory_state.get('ai_profile', {}).get('nickname')

    if ai_nickname:
        system_prefix = f"You are a helpful personal AI assistant named {ai_nickname}."
    else:
        system_prefix = "You are a helpful personal AI assistant."

    if system_content:
        final_system_content = f"{system_prefix} {system_content}"
    else:
        final_system_content = system_prefix

    messages = [
        {
            "role": "system",
            "content": final_system_content
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=1000, # Increased max_new_tokens for more complete responses
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
def should_extract_memory(user_input):
    """
    Uses the LLM to determine if a user's message should trigger memory extraction.
    Returns True if memory extraction is warranted, False otherwise.
    """
    system_prompt = """
You are an AI assistant tasked with identifying if a user's message contains information important enough to be stored in long-term memory. Respond with 'YES' if the message contains factual information about the user, their preferences, relationships, or anything else worth remembering for future interactions. Respond with 'NO' if the message is a greeting, a simple acknowledgment, a question about your capabilities, or trivial conversation.

Examples of 'NO':
- "Hello"
- "How are you?"
- "Thanks"
- "What can you do?"
- "Ok"

Examples of 'YES':
- "My name is Sami."
- "I like apples."
- "My friend John lives in London."
- "I am working on a Python project."
- "Call me Zeus."

Respond with 'YES' or 'NO' only.
"""
    response = chat(user_input, system_content=system_prompt)
    # The LLM response might have extra whitespace or punctuation, so clean it.
    clean_response = response.strip().upper()
    return "YES" in clean_response


In [ ]:
def should_search_web(user_input, memory_context):
    """
    Uses the LLM to decide if the query requires fresh information from the web.
    """
    system_prompt = """
    You are an expert at deciding if a user's request needs current, real-time, or time-sensitive information from the internet.

    Respond 'SEARCH' if the query involves:
    - Current events, news, or sports results (e.g., FIFA 2026, recent matches).
    - Prices, software versions, or tech specs (e.g., RTX 4060 price, latest Android).
    - Weather, stock market, or political figures.
    - Anything that might have changed since 2023.

    Respond 'NO_SEARCH' if the query is:
    - Personal (answered by memory context).
    - General knowledge (e.g., 'What is gravity?').
    - Logic/Math or simple greetings.

    Respond with ONLY 'SEARCH' or 'NO_SEARCH'.
    """

    decision_input = f"Memory Context: {memory_context}\nUser: {user_input}"
    response = chat(decision_input, system_content=system_prompt)
    return "SEARCH" in response.upper()

In [ ]:
def extract_ai_nickname(user_input):
    """
    Uses the LLM to extract a nickname given to the AI by the user.
    Returns the extracted nickname (string) or None if not found.
    """
    system_prompt = """
You are an AI assistant. Your task is to identify if the user is giving you a nickname. If the user assigns you a nickname, extract ONLY the nickname. If no nickname is assigned, respond with 'NONE'.

Examples:
User: "I'll call you Sparky."
Output: "Sparky"

User: "You're my little assistant, Buddy."
Output: "Buddy"

User: "What should I call you?"
Output: "NONE"

User: "You are the best AI."
Output: "NONE"

User: "My new nickname for you is ColabAI."
Output: "ColabAI"

Respond with the extracted nickname OR 'NONE' only.
"""
    response = chat(user_input, system_content=system_prompt)
    clean_response = response.strip()
    if clean_response.upper() != 'NONE':
        return clean_response
    return None


In [ ]:
import json
import os

MEMORY_FILE = "memory.json"

if not os.path.exists(MEMORY_FILE):
    with open(MEMORY_FILE, "w") as f:
        json.dump([], f) # Initialize as an empty list

In [ ]:
DEFAULT_MEMORY_STRUCTURE = {
    "profile": {
        "name": None,
        "preferred_name": None,
        "birthday": None,
        "country": None,
        "hobbies": [],
        "skills": [],
        "siblings": [] # Added siblings field as a list
    },
    "relationships": {
        "friends": [],
        "family": []
    },
    "pets": [], # Will be a list of pet names
    "preferences": {
        "favorite_food": None,
        "favorite_player": None,
        "favorite_color": None,
        "favorite_game": [],
        "favorite_movie": [],
        "favorite_book": []
    },
    "projects": [], # List of project names/descriptions
    "programming_languages": [],
    "interests": [],
    "episodic_memories": [], # For specific events/facts, can be structured further if needed
    "conversation_summaries": [] # For rolling summaries of past conversations
    ,
    "ai_profile": {
        "nickname": None
    }
}

In [ ]:
import json
import re

def extract_memory(user_text):

    memory_system_prompt = """
You are a memory extractor for a personal AI. Your task is to identify and extract important factual information about the user from their messages. The extracted information should be structured as a JSON array of objects, where each object has 'category', 'field', and 'value'.

Only extract information that is significant and long-term. Ignore greetings, small talk, or temporary statements.

Follow these guidelines:
- **Categories**: Use 'profile', 'relationships', 'pets', 'preferences', 'projects', 'programming_languages', 'interests', 'episodic_memories', or 'conversation_summaries'. Choose the most appropriate category.
- **Fields**: Use descriptive, snake_case names for fields within each category (e.g., 'name', 'friends', 'favorite_food'). For list items, the field name should be the plural form (e.g., 'friends', 'pets', 'hobbies').
- **Value**: The value should be the extracted information. For single-value fields (e.g., 'name', 'favorite_food'), the value should be a string. For multi-value fields (e.g., 'friends', 'pets', 'hobbies'), the value should be a JSON array of strings.
- **Multiple items**: If multiple distinct items for a list field are mentioned (e.g., multiple friends or pets), include all of them in the `value` array.
- **Output ONLY the JSON array**. Do not include any other text, explanations, or remarks.
- If no important memory is found, output an empty JSON array: `[]`.

Examples:
User: "My name is Sami and I like programming in Python."
Output:
```json
[
  {"category": "profile", "field": "name", "value": "Sami"},
  {"category": "programming_languages", "field": "programming_languages", "value": ["Python"]}
]
```

User: "My cat's name is Luna and my dog is Max."
Output:
```json
[
  {"category": "pets", "field": "pets", "value": ["Luna", "Max"]}
]
```

User: "I have a friend named Sourav and another friend Tanvir. My two female friends are Nishu and Smrity."
Output:
```json
[
  {"category": "relationships", "field": "friends", "value": ["Sourav", "Tanvir", "Nishu", "Smrity"]}
]
```

User: "Hello!"
Output:
```json
[]
```

User: "Call me Zeus."
Output:
```json
[
  {"category": "profile", "field": "preferred_name", "value": "Zeus"}
]
```
"""

    try:
        raw_response = chat(user_text, system_content=memory_system_prompt)
        # Qwen models might include markdown code block, try to extract JSON from it
        json_match = re.search(r'```json\n([\s\S]*?)\n```', raw_response)
        if json_match:
            json_string = json_match.group(1)
        else:
            # If no markdown block, assume the entire response is JSON
            json_string = raw_response.strip()

        extracted_memories = json.loads(json_string)

        # Basic validation of the extracted structure
        if not isinstance(extracted_memories, list):
            print(f"Warning: LLM returned non-list JSON for memory extraction. Response: {json_string}")
            return []
        for item in extracted_memories:
            if not isinstance(item, dict) or not all(k in item for k in ['category', 'field', 'value']):
                print(f"Warning: LLM returned malformed memory item: {item}")
                return []
        return extracted_memories
    except json.JSONDecodeError as e:
        print(f"Warning: Could not decode JSON from LLM response for memory extraction. Error: {e}\nRaw Response: {raw_response}")
        return []
    except Exception as e:
        print(f"An unexpected error occurred during memory extraction: {e}")
        return []


In [ ]:
def save_memory(new_memory_bullet_points):
    with open(MEMORY_FILE, "r") as f:
        data = json.load(f) # data is now a dictionary

    parsed_new_facts = {}
    for line in new_memory_bullet_points.split("\n"):
        stripped_line = line.strip()
        if stripped_line.startswith('- ') and ':' in stripped_line:
            # Expecting format "- Key: Value"
            key_value = stripped_line[2:].split(':', 1) # Split only on the first colon
            key = key_value[0].strip().replace(' ', '_').lower() # e.g., "User name" -> "user_name"
            value = key_value[1].strip()
            if key and value:
                parsed_new_facts[key] = value

    # Update existing memory with new facts (this will overwrite existing keys)
    data.update(parsed_new_facts)

    with open(MEMORY_FILE, "w") as f:
        json.dump(data, f, indent=2)

In [ ]:
def load_memory():
    with open(MEMORY_FILE, "r") as f:
        return json.load(f) # Returns a dictionary

In [ ]:
def load_memory():
    global DEFAULT_MEMORY_STRUCTURE
    memory = DEFAULT_MEMORY_STRUCTURE.copy()
    for key, value in DEFAULT_MEMORY_STRUCTURE.items():
        if isinstance(value, dict):
            memory[key] = value.copy()
        elif isinstance(value, list):
            memory[key] = []

    if os.path.exists(MEMORY_FILE):
        try:
            with open(MEMORY_FILE, "r") as f:
                loaded_data = json.load(f)
            # Merge loaded data with default structure to ensure all keys exist
            for category, default_values in DEFAULT_MEMORY_STRUCTURE.items():
                if category in loaded_data:
                    if isinstance(default_values, dict):
                        for field, default_value in default_values.items():
                            if field in loaded_data[category]:
                                memory[category][field] = loaded_data[category][field]
                            else:
                                memory[category][field] = default_value # Add missing fields
                    elif isinstance(default_values, list):
                        memory[category] = loaded_data[category] # Lists are overwritten/taken as is
                else:
                    # If a category is missing in loaded data, use its default
                    memory[category] = default_values.copy() if isinstance(default_values, dict) else []
            return memory
        except json.JSONDecodeError:
            print("Warning: memory.json is corrupted or empty. Initializing with default structure.")
            return memory
    else:
        return memory


In [ ]:
def save_memory(memory_object):
    # This function now simply writes the complete memory object
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory_object, f, indent=2)


In [ ]:
def merge_extracted_facts_into_memory(current_memory, extracted_facts):
    """
    Merges extracted facts (list of dicts) into the current structured memory object.
    Handles list merging, single-value updates, and dynamic field creation.
    """
    for fact in extracted_facts:
        category = fact.get('category')
        field = fact.get('field')
        value = fact.get('value')

        if not all([category, field, value is not None]):
            print(f"Warning: Skipping malformed extracted fact: {fact}")
            continue

        if category not in current_memory:
            print(f"Warning: Category '{category}' not found in default memory structure. Skipping fact: {fact}")
            continue

        # Handle top-level list categories directly (e.g., 'pets', 'projects', 'interests')
        if isinstance(current_memory[category], list):
            if isinstance(value, list):
                # Add unique items from extracted value to the existing list
                for item in value:
                    if item not in current_memory[category]:
                        current_memory[category].append(item)
            elif value is not None and value not in current_memory[category]:
                 current_memory[category].append(value)
            continue # Move to the next fact after handling top-level list category

        # Handle nested fields within categories (e.g., 'profile.name', 'relationships.friends')
        if isinstance(current_memory[category], dict):
            # If the field does not exist in the current memory, create it based on the value type
            if field not in current_memory[category]:
                if isinstance(value, list):
                    current_memory[category][field] = []  # Initialize new list fields as empty lists
                else:
                    # For new single-value fields, directly assign the value and move on
                    current_memory[category][field] = value
                    continue # Fact is processed, move to next

            # Now, handle merging/updating based on the *actual type* of the field in current_memory
            # (which might be predefined, or just initialized as a list in the step above).
            if isinstance(current_memory[category][field], list): # Check type of the existing/initialized field
                # It's a list field, merge values
                if isinstance(value, list):
                    for item in value:
                        if item not in current_memory[category][field]:
                            current_memory[category][field].append(item)
                elif value is not None and value not in current_memory[category][field]:
                    current_memory[category][field].append(value)
            else:
                # It's a single-value field (predefined or already set).
                # We only reach here for existing single-value fields that need updating,
                # or if a new field was implicitly treated as a single value and initialized above.
                if value is not None:
                    # Special conflict handling for name/preferred_name
                    if field == 'name' and current_memory[category].get('preferred_name') is None and value != current_memory[category].get('name'):
                        current_memory[category][field] = value
                    elif field == 'name' and current_memory[category].get('preferred_name') is not None and value != current_memory[category].get('preferred_name'):
                        current_memory[category]['preferred_name'] = value
                    elif field == 'preferred_name':
                        current_memory[category][field] = value
                    elif field == 'name' and current_memory[category].get('name') is None:
                        current_memory[category][field] = value
                    elif current_memory[category].get(field) != value: # Only update if different
                        current_memory[category][field] = value

    return current_memory

In [ ]:
def build_prompt(user_input, search_results=None):
    memory_dict = load_memory()
    memory_text_lines = []

    def format_memory_section(section_name, data, indent=0):
        indent_str = "  " * indent
        if isinstance(data, dict):
            if data:
                memory_text_lines.append(f"{indent_str}{section_name.replace('_', ' ').title()}:")
                for key, value in data.items():
                    if value is not None and value != [] and value != {}:
                        if isinstance(value, list):
                            memory_text_lines.append(f"{indent_str}  - {key.replace('_', ' ').title()}: {', '.join(map(str, value))}")
                        else:
                            memory_text_lines.append(f"{indent_str}  - {key.replace('_', ' ').title()}: {value}")
        elif isinstance(data, list):
            if data:
                memory_text_lines.append(f"{indent_str}{section_name.replace('_', ' ').title()}: {', '.join(map(str, data))}")

    for category, content in memory_dict.items():
        format_memory_section(category, content)

    memory_text = "\n".join(memory_text_lines) if memory_text_lines else "No specific facts remembered yet."

    search_context = ""
    if search_results:
        search_context = f"\nFRESH WEB SEARCH RESULTS (Prioritize this for current events):\n{search_results}\n"

    return f"""
Known facts about the user:
{memory_text}
{search_context}
Current message:
{user_input}

Instructions: If web search results are provided above, use them to provide an accurate, up-to-date answer. Cite sources if search was used. If information is still missing, state that you couldn't verify it.
"""

In [ ]:
import json
import os
import torch

MEMORY_FILE = "memory.json"
current_memory = load_memory()
save_memory(current_memory)

print("Chat started. Type 'exit' to stop.")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    # Check for AI nickname first
    ai_nickname_extracted = extract_ai_nickname(user_input)
    if ai_nickname_extracted:
        current_memory = load_memory()
        current_memory['ai_profile']['nickname'] = ai_nickname_extracted
        save_memory(current_memory)
        print(f"AI: Got it! You've named me {ai_nickname_extracted}. I'll remember that!")
        continue

    # SEARCH DECISION LAYER
    search_results = None
    memory_context = str(load_memory())

    force_search = any(kw in user_input.lower() for kw in ["search", "online", "check", "latest", "current", "google"])

    if force_search or should_search_web(user_input, memory_context):
        print("(Searching the web for fresh information...)")
        raw_search = web_search(user_input)
        # Truncate search results to avoid OOM errors (approx 3000 chars)
        search_results = raw_search[:3000] + "... (truncated)" if len(raw_search) > 3000 else raw_search
        # Clear CUDA cache before heavy inference
        torch.cuda.empty_cache()

    # STEP 1: chat response
    prompt = build_prompt(user_input, search_results=search_results)
    try:
        response = chat(prompt)
        print("AI:", response)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("AI: I'm sorry, the search result was too large for me to process. Try asking a more specific question.")
        continue

    # STEP 2: Memory extraction
    if should_extract_memory(user_input):
        extracted_facts = extract_memory(user_input)
        if extracted_facts:
            current_memory = load_memory()
            updated_memory = merge_extracted_facts_into_memory(current_memory, extracted_facts)
            save_memory(updated_memory)
        else:
            print("No significant user memory facts extracted.")